# 03 Features: pool and label per cut-off

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/buildings_clean.parquet`, `data/processed/pv_fr.parquet`.
Output: `data/processed/model_table.parquet` (one row per pool building and cut-off, with features, `y` and `y_2024`).

For each cut-off T in `helpers.CUTOFFS`: pool = buildings without PV by T, not new builds, with enough suitable roof. Label `y` = first PV in the two years after T. For the test split, `y_2024` uses 2024 only.

In [ ]:
import helpers
import pandas as pd

In [ ]:
buildings = pd.read_parquet(helpers.PROCESSED / "buildings_clean.parquet")
pv = pd.read_parquet(helpers.PROCESSED / "pv_fr.parquet")
first_pv = helpers.first_pv_date(pv)
print(len(buildings), "buildings,", len(first_pv), "buildings with a dated PV plant")

Build the pool and the label for each split.

In [ ]:
features = [
    "yield_kwh",
    "suitable_area_m2",
    "roof_area_m2",
    "best_class",
    "n_surfaces",
    "gkat",
    "gklas",
    "gbaup",
    "garea",
    "gastw",
    "ganzwhg",
]
parts = []
for split, cutoff in helpers.CUTOFFS.items():
    start = pd.Timestamp(cutoff)
    end = start + pd.DateOffset(years=helpers.HORIZON_YEARS)
    pool = helpers.build_pool(buildings, first_pv, cutoff)
    part = pool[["egid"] + features].copy()
    part.insert(1, "cutoff", cutoff)
    part.insert(2, "split", split)
    part.insert(3, "y", helpers.make_label(pool["egid"], first_pv, start, end))
    # Robustness label: first PV in the first year only (test split only)
    part["y_2024"] = pd.array([pd.NA] * len(part), dtype="Int64")
    if split == "test":
        end_one_year = start + pd.DateOffset(years=1)
        part["y_2024"] = helpers.make_label(pool["egid"], first_pv, start, end_one_year).astype(
            "Int64"
        )
    parts.append(part)
    print(split, "label window", start.date() + pd.Timedelta(days=1), "to", end.date())

order = ["egid", "cutoff", "split", "y", "y_2024"] + features
model_table = pd.concat(parts, ignore_index=True)[order]
model_table = model_table.sort_values(["cutoff", "egid"]).reset_index(drop=True)

In [ ]:
assert not model_table.duplicated(["egid", "cutoff"]).any()
for split, part in model_table.groupby("split"):
    print(
        split,
        "pool:",
        len(part),
        "positives:",
        int(part["y"].sum()),
        "base rate:",
        round(part["y"].mean(), 4),
    )
test = model_table[model_table["split"] == "test"]
print(
    "test y_2024 positives:",
    int(test["y_2024"].sum()),
    "base rate:",
    round(test["y_2024"].mean(), 4),
)
model_table.to_parquet(helpers.PROCESSED / "model_table.parquet", index=False)